In [ ]:
import numpy as np

def f_2(P):  
    X = P[0]  
    Y = P[1]  
    
    r2 = X**2 + Y**2
    z = (r2)**0.25 * (np.sin(50 * (r2)**0.1)**2 + 1)
    return z



In [8]:
class TSO():
    def __init__(self, function, start_point, lower, upper, k, n_vecinos, sigma, epsilon=0.01, n_max=1000, minimize=True):
        # Parametros del problema
        self.function = function
        self.start_point = np.array(start_point, dtype=float)
        self.lower = np.array(lower)
        self.upper = np.array(upper)
        self.minimize = minimize

        # Parametros del algoritmo
        self.k = k 
        self.n_vecinos = n_vecinos
        self.sigma = sigma
        self.epsilon = epsilon
        self.n_max = n_max

        # Historiales y resultados
        self.hist_t = []
        self.hist_fitness = []
        self.hist_points = []
        self.best_point = None
        self.best_fitness = None

        self.best_iteration = 0
        self.total_neighbors_evaluated=0
        self.total_tabu_neighbors = 0
        self.prop_tabu = 0

        self.distante_to_optimum=None

        self.hist_tabu = []
    def is_tabu(self, x_candidato, tabu_list):
        for x_tabu in tabu_list:
            if np.linalg.norm(x_candidato - x_tabu) < self.epsilon:
                return True
        return False

    def add_to_tabu(self, x, tabu_list):
        """Agrega solución a lista tabú (FIFO)"""
        tabu_list.append(x.copy())
        if len(tabu_list) > self.k:
            tabu_list.pop(0)  # Remueve el primero (el más viejo)

    def fitness_improved(self, f_current):
        if self.minimize:
            return f_current < self.best_fitness
        else:
            return f_current > self.best_fitness

    def run(self):
        curr_point = self.start_point.copy()
        f_curr = self.function(curr_point)
        self.best_point = curr_point.copy()
        self.best_fitness = f_curr
        tabu_list = []

        for t in range(1, self.n_max):
            delta = np.random.normal(
                0, 
                self.sigma, 
                size=(self.n_vecinos, len(curr_point))) #generamos n_vecinos desplazamientos aletorios de longitud curr_point (dimension)
            candidatos = curr_point + delta #es una matriz, cada fila es un punto
            candidatos = np.clip(candidatos, self.lower, self.upper)

            candidatos_validos = []
            fitness_valido = []
            all_neighbors = [] # por si todos son tabú

            for p_candidato in candidatos:
                self.total_neighbors_evaluated += 1
                f_p = self.function(p_candidato) #evaluo el fitnes del candidato
                all_neighbors.append((p_candidato, f_p)) #una tupla, candidato, fitnes
                

                if self.is_tabu(p_candidato, tabu_list) == False:
                    #sino es tabu entonces
                    candidatos_validos.append(p_candidato) #lo agregamos en los validos para luego ver si es tabu
                    fitness_valido.append(f_p)
                else:
                    self.total_tabu_neighbors +=1

            #Selección del mejor vecino
            if len(candidatos_validos) > 0:
                idx = np.argmin(fitness_valido) if self.minimize else np.argmax(fitness_valido) #
                next_point = candidatos_validos[idx].copy() #el mejor de los no tabus
                next_fitness = fitness_valido[idx]
            else:
                best_neighbor = all_neighbors[0]
                for neighbor in all_neighbors:
                    punto = neighbor[0] 
                    fitness = neighbor[1]
                    
                    if self.minimize:
                        if fitness < best_neighbor[1]:
                            best_neighbor = neighbor
                    else:
                        if fitness > best_neighbor[1]:
                            best_neighbor = neighbor

                next_point = best_neighbor[0]
                next_fitness = best_neighbor[1]

            #
            curr_point = next_point.copy() 
            f_curr = next_fitness
            self.add_to_tabu(curr_point, tabu_list) #solo el mejor putno se vuelve tabu
            self.hist_tabu.append([p.copy() for p in tabu_list])

            # 5. Actualizar récord global
            if self.fitness_improved(f_curr):
                self.best_point = curr_point.copy()
                self.best_fitness = f_curr
                self.best_iteration = t

            # Historial
            self.hist_t.append(t)
            self.hist_fitness.append(f_curr)
            self.hist_points.append(curr_point.copy())

            self.prop_tabu = self.total_tabu_neighbors/self.total_neighbors_evaluated

            
        self.distance_to_optimum = np.linalg.norm(self.best_point) #como el optimo es el origen sacamos la norma
        return self.best_point, self.best_fitness

In [9]:
# Definimos el espacio de búsqueda y el punto de inicio
lower_bounds = np.array([-10.0, -10.0])
upper_bounds = np.array([10.0, 10.0])
start_point = [7.5, -6.0]  # Empezamos alejados del origen (0,0)

# Inicializamos tu clase Tabu Search con parámetros recomendados para esta función
ts = TSO(
    function=f_2,
    start_point=start_point,
    lower=lower_bounds,
    upper=upper_bounds,
    k=15,               # Tamaño de la lista Tabú
    n_vecinos=40,       # Cantidad de vecinos por iteración
    sigma=0.5,          # Tamaño del paso aleatorio (desviación estándar)
    epsilon=0.1,        # Radio para considerar a un vecino como Tabú
    n_max=500,          # Iteraciones máximas
    minimize=True       # Buscamos el mínimo global
)

# Ejecutamos el algoritmo
best_x, best_f = ts.run()

# --- Bloque de Impresión de Resultados ---
print("="*40)
print("=== RESULTADOS DEL EXPERIMENTO ===")
print("="*40)
print(f"Mejor punto encontrado (X, Y): {best_x}")
print(f"Mejor fitness alcanzado: {best_f:.6f}")
print(f"Encontrado en la iteración: {ts.best_iteration}")
print(f"Distancia final al óptimo teórico (0,0): {ts.distance_to_optimum:.6f}")
print("-"*40)
print(f"Total de vecinos evaluados: {ts.total_neighbors_evaluated}")
print(f"Total de vecinos rechazados por Tabú: {ts.total_tabu_neighbors}")
print(f"Proporción final de vecinos Tabú: {ts.prop_tabu * 100:.2f}%")

=== RESULTADOS DEL EXPERIMENTO ===
Mejor punto encontrado (X, Y): [ 0.00775956 -0.01328658]
Mejor fitness alcanzado: 0.134475
Encontrado en la iteración: 161
Distancia final al óptimo teórico (0,0): 0.015386
----------------------------------------
Total de vecinos evaluados: 19960
Total de vecinos rechazados por Tabú: 3219
Proporción final de vecinos Tabú: 16.13%
